# 09 - Early Stopping, EMA, and Gradient Clipping

This notebook isolates three training-control callbacks and visualises their
behaviour on synthetic signals: `EarlyStopping` (patience counter and trigger
epoch on a synthetic validation curve), `EMA` (shadow-parameter smoothing of
a noisy parameter trajectory), and `GradientClipper` (fixed vs adaptive
threshold on a synthetic gradient-norm stream).

Modules exercised: `EarlyStopping`, `EMA`, `GradientClipper` from
`pipelines.backbone_pipeline.callbacks`.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy             as np
import torch
import matplotlib.pyplot as plt

SEED   = 0
DEVICE = torch.device("cpu")

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True, warn_only=True)

plt.rcParams.update({
    "figure.dpi"     : 120,
    "savefig.dpi"    : 300,
    "font.size"      : 10,
    "axes.grid"      : True,
    "grid.alpha"     : 0.3,
    "axes.spines.top"   : False,
    "axes.spines.right" : False,
})

print("repo root :", REPO_ROOT)
print("torch     :", torch.__version__)
print("device    :", DEVICE)


In [ ]:
import copy
import torch.nn as nn
from configuration.training_config import TrainerConfig, GaussianConfig, EarlyStoppingConfig, EMAConfig, GradientClipperConfig
from pipelines.backbone_pipeline.callbacks import EarlyStopping, EMA, GradientClipper
from tools import NullLogger, NullTracker

gaussian_cfg = TrainerConfig(gaussian=GaussianConfig(n_default_gaussians=2, x_min=-10.0, x_max=40.0))
base_cfg     = gaussian_cfg
nl, nt       = NullLogger(), NullTracker()


## Early stopping on a synthetic validation curve

We feed a validation loss that decreases, reaches a minimum, then drifts
upward. With a fixed patience the callback should fire a number of epochs
after the minimum equal to the patience, and report the best epoch.

In [ ]:
cfg = copy.deepcopy(base_cfg)
cfg.early_stopping = EarlyStoppingConfig(patience=8, min_delta=1e-3, restore_best=False)
es = EarlyStopping(cfg, nl, nt)

model = nn.Linear(2, 2)
MIN_EPOCH = 25
epochs = np.arange(60)
descent = np.exp(-epochs[:MIN_EPOCH] / 8.0)
ascent  = descent[-1] + 0.03 * (epochs[MIN_EPOCH:] - MIN_EPOCH)
val     = 0.2 + np.concatenate([descent, ascent])

counters, stopped_epoch = [], None
for e in epochs:
    stop = es(float(val[e]), model, int(e))
    counters.append(es.counter)
    if stop and stopped_epoch is None:
        stopped_epoch = int(e)
        break

fig, ax1 = plt.subplots(figsize=(7, 4))
n = len(counters)
ax1.plot(epochs[:n], val[:n], color='C0', label='val loss')
ax1.axvline(es.best_epoch, color='green', ls='--', lw=1, label=f'best epoch = {es.best_epoch}')
if stopped_epoch is not None:
    ax1.axvline(stopped_epoch, color='red', ls='--', lw=1, label=f'stop epoch = {stopped_epoch}')
ax1.set_xlabel('epoch'); ax1.set_ylabel('val loss', color='C0')
ax2 = ax1.twinx()
ax2.plot(epochs[:n], counters, color='C1', label='patience counter'); ax2.grid(False)
ax2.set_ylabel('patience counter', color='C1')
ax1.set_title('Early stopping trigger logic')
ax1.legend(loc='upper center', frameon=False)
fig.tight_layout()
plt.show()
print('best epoch:', es.best_epoch, ' stop epoch:', stopped_epoch, ' patience:', es.patience)


## EMA smoothing of a parameter trajectory

We simulate a single noisy parameter wandering during training and apply the
`EMA.update` rule each step. The shadow value should lag and smooth the raw
trajectory; a higher decay produces a slower, smoother shadow.

In [ ]:
def ema_trace(decay, raw):
    cfg = copy.deepcopy(base_cfg)
    cfg.ema = EMAConfig(use_ema=True, ema_decay=decay)
    ema = EMA(cfg, nl, nt)
    m = nn.Linear(1, 1, bias=False)
    ema.init(m)
    shadow = []
    for v in raw:
        with torch.no_grad():
            m.weight.fill_(float(v))
        ema.update(m)
        shadow.append(float(next(iter(ema.shadow.values())).item()))
    return shadow

torch.manual_seed(SEED)
steps = 300
raw = 1.0 - np.exp(-np.linspace(0, 4, steps)) + 0.05 * np.random.randn(steps)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(raw, color='lightgrey', lw=1, label='raw parameter')
for decay in [0.9, 0.99, 0.999]:
    ax.plot(ema_trace(decay, raw), label=f'EMA decay={decay}')
ax.set_xlabel('optimizer step')
ax.set_ylabel('parameter value')
ax.set_title('EMA shadow vs raw parameter')
ax.legend(frameon=False)
fig.tight_layout()
plt.show()


## Gradient clipping thresholds

We feed a synthetic gradient-norm stream with occasional spikes through the
clipper history and plot the fixed threshold against the adaptive
percentile and mean+k*std thresholds, which track the recent distribution.

In [ ]:
torch.manual_seed(SEED)
n_steps = 400
grad_norms = np.abs(0.5 + 0.1 * np.random.randn(n_steps))
grad_norms[::40] += np.linspace(1.0, 4.0, len(grad_norms[::40]))

def adaptive_thresholds(mode, **over):
    cfg = copy.deepcopy(base_cfg)
    cfg.gradient_clipper = GradientClipperConfig(clip_mode=mode, adaptive_window=50, **over)
    gc = GradientClipper(cfg, nl, nt)
    out = []
    for v in grad_norms:
        gc.history.append(float(v))
        out.append(gc._compute_adaptive_threshold())
    return out

pct_thr = adaptive_thresholds('adaptive_percentile', adaptive_percentile=95.0)
std_thr = adaptive_thresholds('adaptive_mean_std',   adaptive_mean_std_k=2.0)

fig, ax = plt.subplots(figsize=(7.5, 4))
ax.plot(grad_norms, color='lightgrey', lw=1, label='gradient norm')
ax.axhline(1.0, color='C3', ls='--', lw=1, label='fixed threshold = 1.0')
ax.plot(pct_thr, color='C0', label='adaptive percentile (95)')
ax.plot(std_thr, color='C1', label='adaptive mean + 2*std')
ax.set_xlabel('optimizer step')
ax.set_ylabel('gradient norm / threshold')
ax.set_title('Gradient clipping thresholds')
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
plt.show()


## Expected visual outcome

Early stopping marks the best epoch at the validation minimum and fires the
stop exactly `patience` epochs later, with the counter ramping up over that
window. The EMA shadows lag the raw trajectory, with larger decay giving a
smoother, slower-moving shadow. The gradient-clipping panel shows the fixed
threshold as a flat line while the adaptive thresholds rise and fall with the
recent norm distribution, sitting above the bulk of the values but reacting
to the spikes.